In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [ ]:
path = 'data/current_data/test2/SMOC_20260204_R20260218.nc'

In [ ]:
ds = xr.open_dataset(path).sel(longitude=slice(26,28), latitude=slice(-35.5, -33.5), time='2026-02-04T04:30:00.000000000').load()
ds

In [ ]:
plt.imshow(ds['uo'].values.squeeze(), aspect='auto')

In [ ]:
transform = from_origin(
    west=ds.longitude.min().item(),
    north=ds.latitude.min().item(),
    xsize=(ds.longitude[1] - ds.longitude[0]).item(),
    ysize=(ds.latitude[0] - ds.latitude[1]).item(),
)

In [ ]:
data = ds[['uo', 'vo']].to_array().values.squeeze()

with rasterio.open(
    "data/current_data/test2/ocean_current.tif",
    "w",
    driver="GTiff",
    height=data.shape[1],
    width=data.shape[2],
    count=data.shape[0],
    dtype=data.dtype,
    crs="EPSG:4326",  # or your CRS
    transform=transform,
) as dst:
    dst.write(data)

In [ ]:
src = rasterio.open('data/current_data/test2/ocean_current.tif')
uo =  src.read(1)
vo = src.read(2)
uo[uo==-1000]=np.nan
vo[vo==-1000]=np.nan

In [ ]:
plt.imshow(uo)

In [ ]:
def expected_doppler_biomass(u_e, u_n):#, heading_deg, incidence_deg, freq_hz=435e6):
    c = 299792458.0
    lam = c / 435e6

    H = np.deg2rad(-12.9071172)
    th = np.deg2rad(26.9803)

    # right-looking radar
    g_e = np.cos(H)
    g_n = -np.sin(H)

    u_look = u_e * g_e + u_n * g_n
    v_los = u_look * np.sin(th)
    f_d = -2.0 * v_los / lam
    return f_d

In [ ]:
exp = expected_doppler_biomass(uo, vo)

In [ ]:
plt.imshow(exp)
plt.colorbar()